# Pos/neg feature gap -- does K5's feature already separate the accident from the normal part of its own video?

Rule: `core/docs/final/POSNEG_GAP.md`, written 2026-11-11 **before any number**. **Inference only on K5 s2024, nothing is trained, K5 stays the main model.** Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

**Gate (N-label only):** STOP the video-only hard-negative idea iff `mean(gap) < 1.5 x mean(floor)`; otherwise CONTINUE. N-thr, N-soft, precision / recall and attention masses are printed, never gated. n = 1 checkpoint: the verdict reads this checkpoint only. Ground-truth frame labels (D-N1) are used to *measure*; a training loss on Nexar has only video-level labels.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K5 ckpt + J10, Nexar caches, census) | no | 2-3 min |
| 4 | tests on the tool -- **raises if red** | no | 1 min |
| 5 | **the read**: K5 on every positive video of `nexar_val` | yes (CPU works, slower) | 5-15 min |
| 6 | report -> **paste `REPORTS/pos_neg_gap/pos_neg_gap.md` to Claude** | no | seconds |

Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/kim/REPORTS/pos_neg_gap/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, ckpts
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar caches, census, K5's statistics
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5; this notebook's outputs go here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/pos_neg_gap.py').is_file(), 'this upload has no pos_neg_gap -- re-upload kat-vad/'
assert (REPO / 'core/docs/final/POSNEG_GAP.md').is_file(), 'no POSNEG_GAP.md (the rule) -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert constants.POSNEG_GAP_RATIO == 1.5, f'the bar is fixed at 1.5, found {constants.POSNEG_GAP_RATIO}'

K5      = constants.KIM_MAIN_ARM                  # 'K5'
SEED    = constants.KIM_SEED                      # 2024
ENC     = constants.V5_ENCODER                    # V2-S
SPLIT   = constants.V2_SPLIT_NEXAR_VAL            # nexar_test is never read here
V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'

KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME     # Thesis-Final/outputs/kim
K5_RUN   = KIM / K5 / f's{SEED}'
CKPT     = K5_RUN / 'checkpoint_last.pt'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'      # K5 (Nexar)'s statistics: the same as its Nexar read
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
NEXAR_DEFS = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'
OUT      = KIM / 'REPORTS' / constants.POSNEG_REPORT_STEM
REPORT   = OUT / f'{constants.POSNEG_REPORT_STEM}.md'
READOUT  = OUT / f'{constants.POSNEG_REPORT_STEM}.json'
LOCAL    = Path('/content/pos_neg_gap')
LOCAL.mkdir(parents=True, exist_ok=True)
print(f'repo {REPO}\nK5 ckpt {CKPT}\nout {OUT}')


In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 5 runs on CPU, slower'


In [ ]:
import torch

from core.data.v2_splits import load_split  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
    drive.mount('/content/drive')


# ---- preflight: every input exists, K5's checkpoint is finished (J10), the caches are K5's own ----
k5_step = json.loads((K5_RUN / 'verified.json').read_text())['step']
last = json.loads((K5_RUN / 'metrics.jsonl').read_text().strip().splitlines()[-1])['global_step']
assert last == k5_step, f'K5 J10: verified step {k5_step} != metrics last step {last}'
census = json.loads(CENSUS.read_text())['videos']
NEXAR_IDS = sorted(load_split(SPLIT))
POS_IDS = [v for v in NEXAR_IDS if census[v]['label'] == 1]
for name, path in (('K5 ckpt', CKPT), ('census', CENSUS), ('Nexar defs', NEXAR_DEFS / constants.DEFS_FILENAME),
                   ('K5 statistics', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME)):
    assert path.exists(), f'missing {name}: {path}'
for name, d in (('Nexar CLIP', NEXAR_X), ('Nexar V2-S', NEXAR_U)):
    miss = [v for v in POS_IDS if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} positive videos have no {name} row: {miss[:5]}'
m = json.loads((A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'K5 statistics are {m}'
print(f'preflight OK: K5 step {k5_step} | {SPLIT}: {len(NEXAR_IDS)} videos, {len(POS_IDS)} positive')


## 4. Tests on the tool (CPU) -- **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_pos_neg_gap.py'])


## 5. The read (GPU if there is one)

One checkpoint (K5 s2024), every positive video of `nexar_val`, whole video at the N-B stride. Per video: `pos` (attention over all frames, `+logit`, tau 0.02), `neg` of three kinds (GT-normal frames, `sigmoid < 0.5`, all frames, weights `-logit`), the gap `1 - cos`, and the floor (the same two pools on the GT-normal frames only). The tool re-checks J10 on the checkpoint and K4 on the statistics itself.


In [ ]:
if READOUT.exists():
    print(f'on Drive: {READOUT}')
else:
    if not torch.cuda.is_available():
        print('WARNING: torch sees no GPU -- the read runs on CPU')
    local = LOCAL / 'out'
    shutil.rmtree(local, ignore_errors=True)
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.pos_neg_gap', '--name', K5, '--ckpt', CKPT,
         '--census', CENSUS, '--clip-s1-dir', stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', POS_IDS),
         '--video-s1-dir', stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', POS_IDS),
         '--stats-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', []),
         '--data-dir', stage(NEXAR_DEFS, LOCAL / 'stage' / 'nexar_defs'),
         '--seed', SEED, '--out-dir', local], log=LOCAL / 'pos_neg_gap.log')
    publish(local, OUT)
    flush_drive()
    print(f'read: {(time.time() - t0) / 60:.1f} min')


## 6. Report: gate + verdict -> **paste `REPORTS/pos_neg_gap/pos_neg_gap.md` to Claude**

In [ ]:
r = json.loads(READOUT.read_text())
assert r['global_step'] == k5_step, f"read is step {r['global_step']}, K5 is {k5_step}"
assert r['kim_on'], 'the checkpoint built without KIM -- this is not K5'
assert r['split'] == SPLIT, r['split']
assert r['videos_scored'] + len(r['videos_skipped']) == len(POS_IDS), (
    r['videos_scored'], len(r['videos_skipped']), len(POS_IDS))
print(REPORT.read_text())
print(f">>> decision {r['decision']}: gap(N-label) / floor = {r['ratio_gap_label_over_floor']} against the bar {r['ratio_bar']} (STOP iff below)")
